In [ ]:
import xarray as xr
import subprocess
import glob
import os

In [ ]:
# Abrimos ambos
nuevo = xr.open_dataset('../salidasWRF2019/wrfout_d01_2019-02-22_15:58:00')
viejo = xr.open_dataset('/home/gcamilo/serafin/wrfout_d01_2018-12-11_17_30_00')

# 1) Comparar variables presentes
vars_nuevo = set(nuevo.data_vars)
vars_viejo = set(viejo.data_vars)

print("Variables solo en el nuevo:", vars_nuevo - vars_viejo)
print("Variables solo en el viejo:", vars_viejo - vars_nuevo)
print("Variables en común:", len(vars_nuevo & vars_viejo))

# 2) Comparar dimensiones
print("\nDims nuevo:", dict(nuevo.dims))
print("Dims viejo:", dict(viejo.dims))

# 3) Comparar atributos globales (config del run: physics, dominio, etc)
attrs_nuevo = nuevo.attrs
attrs_viejo = viejo.attrs

claves_distintas = {k for k in attrs_nuevo if attrs_nuevo.get(k) != attrs_viejo.get(k)}
print("\nAtributos globales distintos:")
for k in claves_distintas:
    print(f"  {k}: viejo={attrs_viejo.get(k)!r} | nuevo={attrs_nuevo.get(k)!r}")

# 4) Chequeo específico: variables de ELEC (lightning flash density, etc)
elec_vars = [v for v in nuevo.data_vars if 'FLSH' in v.upper() or 'ELEC' in v.upper() or 'LTG' in v.upper()]
print("\nVariables relacionadas a ELEC en el nuevo archivo:", elec_vars)

In [ ]:
import glob, os, xarray as xr
raiz = "/serafin/gcamilo/Trabajo_Final"
print([d for d in os.listdir(raiz) if "salida" in d.lower()])

for mes in ["2019-01", "2019-02"]:
    fs = sorted(glob.glob(f"{raiz}/salidasWRF*/wrfout_d01_{mes}*"))
    if not fs:
        print(mes, "-> sin archivos")
        continue
    a = xr.open_dataset(fs[0]).attrs
    print(mes, len(fs), "archivos |", os.path.dirname(fs[0]), "|",
          a["CEN_LAT"], a["CEN_LON"])

In [ ]:
posibles = [v for v in nuevo.data_vars if any(s in v.upper() for s in ['FLSH','FLASH','LTG','LIGHT','IC_FLASH','CG_FLASH'])]
print(posibles)

In [ ]:
for v in ['LIGHT', 'LIGHTDENS', 'LIGHTDIS']:
    print(v, '-', nuevo[v].attrs.get('description', 'sin descripción'), '|', nuevo[v].attrs.get('units', 'sin unidades'))

In [ ]:
archivo_test = "../salidasWRF/wrfout_d01_2019-02-24_00:04:00"
ds_test = xr.open_dataset(archivo_test)
print(ds_test)
print("\n--- Intentando leer LIGHTDENS ---")
print(ds_test["LIGHTDENS"].values)

In [ ]:
# Variables que NUNCA deberían ser 0 en un output válido
print("T (temperatura perturbación) - min/max:", ds_test["T"].values.min(), ds_test["T"].values.max())
print("PSFC (presión superficie) - min/max:", ds_test["PSFC"].values.min(), ds_test["PSFC"].values.max())
print("PH (geopotencial perturbación) - min/max:", ds_test["PH"].values.min(), ds_test["PH"].values.max())

In [ ]:
base_local = "../salidasWRF"
fechas = ["2018-12-14", "2018-12-15", "2019-01-02", "2019-01-03", "2019-02-24", "2019-02-25"]

for fecha in fechas:
    archivos = glob.glob(os.path.join(base_local, f"wrfout_d01_{fecha}*"))
    print(f"{fecha}: {len(archivos)} archivos")

In [ ]:
import xarray as xr
import glob

# Agarramos el primer archivo de la tanda nueva de enero 2019
archivos_nuevos = sorted(glob.glob("../salidasWRF/wrfout_d01_2019-01-09*"))
ds = xr.open_dataset(archivos_nuevos[0])

# Extraemos las dimensiones espaciales
ancho = ds.sizes.get('west_east')
alto = ds.sizes.get('south_north')

print(f"Dimensiones de la grilla (píxeles): {ancho} (ancho) x {alto} (alto)")
print(f"Total de píxeles por capa: {ancho * alto}")

# Si querés ver TODAS las dimensiones (niveles verticales, tiempo, etc.):
print("\nTodas las dimensiones del archivo:")
print(ds.dims)

In [ ]:
claves = [k for k in nuevo.attrs
          if any(s in k.upper() for s in ['ELEC', 'INDUC', 'CHRG', 'CHARGE', 'LIGHT', 'MP_PHYS'])]
for k in sorted(claves):
    print(k, '=', nuevo.attrs[k])
print('total de atributos:', len(nuevo.attrs))